# FastLMM GWAS 

This notebook runs a single-SNP genome-wide association study (GWAS) with [FastLMM](https://fastlmm.github.io/FaST-LMM/), while excluding all variants on chromosome 17 from both association testing and kinship estimation.

The notebook produces:

- a CSV containing the complete GWAS results;
- a CSV containing variants below a specified p-value threshold;
- a Manhattan plot; and
- a Q–Q plot.

> **Important:** The p-value threshold in this notebook is derived from a permutation test. The genotype

## 1. Input requirements

The genotype data must be a PLINK binary dataset with matching `.bed`, `.bim`, and `.fam` files. `BED_PREFIX` should contain the shared path and filename **without** one of these extensions.

The phenotype and covariate files must use a FastLMM/PLINK-compatible format:

- the first two columns identify the family and individual (`FID` and `IID`);
- phenotype values follow those identifier columns; and
- covariates must not contain missing values.

FastLMM intersects and reorders individuals using their identifiers. Nevertheless, confirm that genotype, phenotype, and covariate identifiers use the same naming convention before running the analysis.

## 2. Environment and imports

Install the required packages in the active Jupyter environment if they are not already available. The installation line is commented out so the notebook does not modify an environment unexpectedly.

In [ ]:
# Uncomment and run once if these packages are not installed.
# %pip install fastlmm pysnptools numpy pandas matplotlib


In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from fastlmm.association import single_snp
from fastlmm.util import util as fastlmm_util
from fastlmm.util.stats import plotp
from pysnptools.snpreader import Bed

logging.basicConfig(level=logging.INFO)


## 3. Analysis configuration

Replace the placeholder paths below. Set `COVARIATE_FILE = None` if the analysis does not use covariates.

In [ ]:
# PLINK file prefix: for data/genotypes.bed, use Path("data/genotypes").
BED_PREFIX = Path("path/to/plink_dataset")
PHENOTYPE_FILE = Path("path/to/phenotype.txt")
COVARIATE_FILE = Path("path/to/covariates.txt")  # Set to None if unused.

OUTPUT_DIRECTORY = Path("path/to/output")
FULL_RESULTS_FILE = OUTPUT_DIRECTORY / "gwas_results_without_chr17.csv"
THRESHOLDED_RESULTS_FILE = OUTPUT_DIRECTORY / "snps_below_pvalue_threshold_without_chr17.csv"
MANHATTAN_PLOT_FILE = OUTPUT_DIRECTORY / "manhattan_without_chr17.png"
QQ_PLOT_FILE = OUTPUT_DIRECTORY / "qq_plot_without_chr17.png"

EXCLUDED_CHROMOSOME = 17
PVALUE_THRESHOLD = 0.012114775
GB_GOAL = 2
COUNT_A1 = True


## 4. Validate the configured files

This cell checks for the three PLINK files, the phenotype file, and the optional covariate file before starting a potentially long GWAS run.

In [ ]:
# Accept a BED_PREFIX that was accidentally entered with the .bed extension.
if BED_PREFIX.suffix.lower() == ".bed":
    BED_PREFIX = BED_PREFIX.with_suffix("")

required_files = [
    Path(f"{BED_PREFIX}.bed"),
    Path(f"{BED_PREFIX}.bim"),
    Path(f"{BED_PREFIX}.fam"),
    PHENOTYPE_FILE,
]
if COVARIATE_FILE is not None:
    required_files.append(COVARIATE_FILE)

missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    missing_list = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Required input file(s) not found:\n{missing_list}")

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Input validation passed.")
print(f"Output directory: {OUTPUT_DIRECTORY.resolve()}")


## 5. Load genotypes and exclude chromosome 17

In [ ]:
snp_reader = Bed(str(BED_PREFIX), count_A1=COUNT_A1)
chromosome_values = snp_reader.pos[:, 0]
excluded_mask = chromosome_values == float(EXCLUDED_CHROMOSOME)

if not np.any(excluded_mask):
    available = ", ".join(f"{value:g}" for value in np.unique(chromosome_values))
    raise ValueError(
        f"Chromosome {EXCLUDED_CHROMOSOME} was not found. "
        f"Available chromosome values: {available}"
    )

included_mask = ~excluded_mask
test_snps = snp_reader[:, included_mask]
kinship_snps = test_snps

print(f"Individuals in PLINK data: {snp_reader.iid_count:,}")
print(f"Total SNPs: {snp_reader.sid_count:,}")
print(f"Excluded chromosome-{EXCLUDED_CHROMOSOME} SNPs: {excluded_mask.sum():,}")
print(f"SNPs retained for GWAS: {test_snps.sid_count:,}")


## 6. Run the GWAS

In [ ]:
covariate_argument = None if COVARIATE_FILE is None else str(COVARIATE_FILE)

results = single_snp(
    test_snps=test_snps,
    pheno=str(PHENOTYPE_FILE),
    K0=kinship_snps,
    covar=covariate_argument,
    leave_out_one_chrom=True,
    GB_goal=GB_GOAL,
    count_A1=COUNT_A1,
)

print(f"GWAS completed for {len(results):,} SNPs.")
results.head()


## 7. Save complete and thresholded results

In [ ]:
required_result_columns = {"Chr", "ChrPos", "PValue"}
missing_columns = required_result_columns.difference(results.columns)
if missing_columns:
    raise KeyError(f"FastLMM results are missing columns: {sorted(missing_columns)}")

results.to_csv(FULL_RESULTS_FILE, index=False)

thresholded_results = (
    results.loc[results["PValue"] < PVALUE_THRESHOLD]
    .sort_values("PValue")
    .copy()
)
thresholded_results.to_csv(THRESHOLDED_RESULTS_FILE, index=False)

print(f"Complete results: {FULL_RESULTS_FILE}")
print(
    f"SNPs with PValue < {PVALUE_THRESHOLD:g}: "
    f"{len(thresholded_results):,}"
)
print(f"Thresholded results: {THRESHOLDED_RESULTS_FILE}")
thresholded_results.head()


## 8. Manhattan and Q–Q plots

The Manhattan plot shows association strength across the retained chromosomes. The horizontal line marks `PVALUE_THRESHOLD`.

The Q–Q plot compares observed and expected p-value distributions. Broad deviation from the diagonal can reflect population structure, relatedness, technical artifacts, or true polygenic signal and should be investigated rather than interpreted automatically.

In [ ]:
plt.figure(figsize=(12, 7))
fastlmm_util.manhattan_plot(
    results[["Chr", "ChrPos", "PValue"]].to_numpy(),
    pvalue_line=PVALUE_THRESHOLD,
    xaxis_unit_bp=False,
)
plt.title(f"FastLMM GWAS excluding chromosome {EXCLUDED_CHROMOSOME}")
plt.tight_layout()
plt.savefig(MANHATTAN_PLOT_FILE, dpi=300, bbox_inches="tight")
plt.show()
print(f"Manhattan plot: {MANHATTAN_PLOT_FILE}")


In [ ]:
plt.figure(figsize=(7, 7))
plotp.qqplot(
    results["PValue"].to_numpy(),
    xlim=[0, 5],
    ylim=[0, 5],
)
plt.title(f"Q–Q plot excluding chromosome {EXCLUDED_CHROMOSOME}")
plt.tight_layout()
plt.savefig(QQ_PLOT_FILE, dpi=300, bbox_inches="tight")
plt.show()
print(f"Q–Q plot: {QQ_PLOT_FILE}")


See the [FastLMM `single_snp` documentation](https://fastlmm.github.io/FaST-LMM/#fastlmm.association.single_snp) and the [FaST-LMM repository](https://github.com/fastlmm/FaST-LMM) for API details and methodological references.